# 02 · Limpieza y análisis exploratorio (EDA)
Partimos del tablón analítico que dejó el notebook 01 (una fila por producto, tienda y día). Primero resolvemos los nulos de `sell_price` y después miramos qué patrones tienen las ventas.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

RUTA = (Path("..") / "02_datos" / "03_Entrenamiento").resolve()
tablon = pd.read_parquet(RUTA / "tablon_analitico.parquet")
print(tablon.shape)
tablon.dtypes

## 1. Limpieza: nulos de `sell_price`
Antes de decidir qué hacer con un nulo hay que entender **por qué** está. Miramos qué filas son.

In [ ]:
sin_precio = tablon["sell_price"].isna()

tablon.loc[sin_precio].groupby(["item_id", "store_id"]).agg(
    filas=("d", "size"),
    desde=("date", "min"),
    hasta=("date", "max"),
    ventas=("ventas", "sum"),
)

Los nulos son de **un solo producto** (`FOODS_3_288`), en las primeras semanas de 2013 y con **0 ventas**. Ese producto todavía no estaba a la venta en esas tiendas.

Esas filas no son "días en que no se vendió nada", son "días en que el producto no existía". Si las dejamos, el modelo aprende ceros falsos. Son solo 71 filas, así que las **eliminamos**. Un `assert` confirma que no perdemos ninguna venta real.

In [ ]:
assert (tablon.loc[sin_precio, "ventas"] == 0).all()

tablon_limpio = tablon.loc[~sin_precio].reset_index(drop=True)

# Los nulos de eventos significan "ese día no hubo evento"
cols_evento = ["event_name_1", "event_type_1", "event_name_2", "event_type_2"]
tablon_limpio[cols_evento] = tablon_limpio[cols_evento].fillna("Ninguno")
tablon_limpio["hay_evento"] = tablon_limpio["event_name_1"] != "Ninguno"

print(f"{len(tablon)} -> {len(tablon_limpio)} filas | nulos restantes: {int(tablon_limpio.isna().sum().sum())}")
tablon_limpio.to_parquet(RUTA / "tablon_limpio.parquet", index=False)

## 2. EDA: ¿cómo se comportan las ventas?
### Evolución mensual
Suma de unidades vendidas de las 20 series, mes a mes. Buscamos **tendencia** y **estacionalidad**.

In [ ]:
mensual = tablon_limpio.groupby(tablon_limpio["date"].dt.to_period("M"))["ventas"].sum()
mensual.index = mensual.index.to_timestamp()

fig, ax = plt.subplots(figsize=(12, 4))
mensual.plot(ax=ax, marker="o")
ax.set_title("Unidades vendidas por mes")
ax.set_ylabel("Unidades")
plt.show()

**Observación:** diciembre es el mes más bajo de cada año (13 a 14 mil unidades) y julio a septiembre los más altos. El total anual baja de 218 mil (2013) a 208 mil (2014) y 202 mil (2015). Septiembre de 2013 (26,4 mil) se destaca como pico y habría que revisarlo. La caída de diciembre puede deberse a días de cierre; queda por verificar.

### Por tienda y por producto
¿Quién vende más? Si una tienda o un producto domina, el modelo tiene que tratarlo distinto.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

tablon_limpio.groupby("store_id")["ventas"].sum().plot.bar(ax=axes[0], rot=0)
axes[0].set_title("Ventas totales por tienda")

tablon_limpio.groupby("item_id")["ventas"].sum().sort_values().plot.barh(ax=axes[1])
axes[1].set_title("Ventas totales por producto")

plt.tight_layout()
plt.show()

**Observación:** `CA_3` concentra el **80 %** de las ventas (503 mil unidades contra 125 mil de `CA_4`). Entre los productos, `FOODS_3_090` pesa el 26 % y los tres primeros (`090`, `586`, `120`) suman el 54 %. El volumen es muy desparejo entre series.

### Día de la semana
Promedio de unidades por fila (producto y tienda) según el día.

In [ ]:
orden = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=tablon_limpio, x="weekday", y="ventas", order=orden, errorbar=None, ax=ax)
ax.set_title("Venta promedio por día de la semana")
ax.set_xlabel("")
plt.show()

**Observación:** sábado y domingo venden unas **39 % más** que el miércoles (34,7 y 34,4 contra 24,9 unidades). Hay una estacionalidad semanal clara, y la columna `weekday` va a ser una variable importante para el modelo.

### Efecto de los eventos
¿Se vende distinto en días con evento (feriados, fiestas) que en días normales?

In [ ]:
print(tablon_limpio.groupby("hay_evento")["ventas"].mean().round(1))
print()
print(tablon_limpio.groupby("event_type_1")["ventas"].mean().round(1).sort_values(ascending=False))

**Observación:** en promedio, un día con evento vende lo mismo que uno sin evento (28,6 vs 28,8 unidades). Los eventos no mueven las ventas a este nivel agregado. Con solo 20 series, un efecto puntual en un producto puede quedar tapado por el promedio.

## Resumen
- Se eliminaron 71 filas de un producto que todavía no estaba a la venta. No se perdió ninguna venta real.
- Hay estacionalidad **semanal** (fin de semana arriba) y **anual** (diciembre abajo, mediados de año arriba).
- Las ventas están muy concentradas en una tienda y en pocos productos.
- Los eventos no muestran un efecto visible en el promedio.

Datos limpios en `02_datos/03_Entrenamiento/tablon_limpio.parquet`. Siguiente paso: construir variables (rezagos, medias móviles) y entrenar un primer modelo.